# **Importing necessary libraries**


In [ ]:
# Import necessary libraries for image processing and feature detection
from scipy.spatial import distance
from imutils import face_utils
import imutils
import dlib
import cv2

# Import libraries for data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import os

# Import Google Colab library for mounting Google Drive
from google.colab import drive

# Mount Google Drive to access files
drive.mount('/content/drive')

# Set the directory for the data
data_dir = 'drive/MyDrive/VIC Project/train/tired/'


# **Reading the UTA-RLDD frames in order**

In [ ]:
import os

def display_first_four_characters(folder_path):
    # Check if the folder exists
    first_four_chars = []

    if not os.path.exists(folder_path):
        print(f"The folder '{folder_path}' does not exist.")
        return

    # List all files in the folder
    files = os.listdir(folder_path)

    # Display the first four characters for each file
    for file_name in files:
        first_four_chars.append(str(file_name[:4]))

    # Remove duplicates by converting the list to a set and back to a list
    first_four_chars = list(set(first_four_chars))

    return first_four_chars

In [ ]:
import random
List=display_first_four_characters(folder_path)
List=np.random.choice(List,100)

In [ ]:
import os

def display_files_ordered(folder_path, number):
    # Check if the folder exists
    if not os.path.exists(folder_path):
        print(f"The folder '{folder_path}' does not exist.")
        return

    # List all files in the folder
    files = os.listdir(folder_path)

    # Filter files that start with the specified numeric prefix
    number_files = [file for file in files if file.startswith(number)]

    # Sort files based on the numeric part after the prefix
    number_files_sorted = sorted(number_files, key=lambda x: int(x[10]) if len(x) == 15 else int(x[10:12]))

    return number_files_sorted


# **Detecting facial landmarks**

In [ ]:
# Initialize a face detector using dlib's built-in frontal face detector
detect = dlib.get_frontal_face_detector()

# Initialize a shape predictor using the dilb pre-trained model for facial landmarks
predict = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

In [ ]:
# Define the index range for the left eye landmarks in the facial landmarks dictionary
(lStart, lEnd) = face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]

# the right eye
(rStart, rEnd) = face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]

# the mouth
(mStart, mEnd) = face_utils.FACIAL_LANDMARKS_IDXS["mouth"]

# the jaw
(jStart, jEnd) = face_utils.FACIAL_LANDMARKS_IDXS["jaw"]


In [ ]:
from skimage.io import imread

def image_landmarks(number):
    # Initialize variables to store convex hulls and a counter
    leftEyeHull, rightEyeHull, mouthHull, jawHull, jint = [], [], [], [], []
    c = 0

    # Retrieve the list of sorted files based on the provided numeric prefix
    frames = display_files_ordered(folder_path, number)

    # Loop until a face is detected in an image
    subjects = []
    while len(subjects) == 0:
        # Read the image and resize it
        img1 = imread(os.path.join(data_dir, frames[c]))
        frame = cv2.resize(img1, (450, 800))

        # Convert the resized image to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Detect faces in the grayscale image
        subjects = detect(gray, 0)
        c += 1

    # Iterate over detected faces
    for subject in subjects:
        # Predict facial landmarks
        shape = predict(frame, subject)
        shape = face_utils.shape_to_np(shape)  # Convert to NumPy Array

        # Extract specific facial landmarks for left eye, right eye, mouth, and jaw
        leftEye = np.array([shape[i] for i in [42, 43, 45, 47]])
        rightEye = np.array([shape[i] for i in [36, 38, 39, 40]])
        mouth = shape[48:59:3]
        jaw = shape[jStart:jEnd]

        # Compute convex hulls for each facial region
        leftEyeHull = cv2.convexHull(leftEye)
        rightEyeHull = cv2.convexHull(rightEye)
        mouthHull = cv2.convexHull(mouth)
        jawHull = cv2.convexHull(jaw)

    # Return the computed convex hulls, the index where a face was detected
    return leftEyeHull, rightEyeHull, mouthHull, jawHull, jint, c - 1

# **Defining the optical flow algorithms**

In [ ]:
# Sobel filter for detecting horizontal edges
mask_x = np.array([[-1, 1],
                   [-1, 1]])

# Sobel filter for detecting vertical edges
mask_y = np.array([[-1, -1],
                   [1, 1]])

# Gradient filter for detecting edges in one direction
mask_t1 = np.array([[-1, -1],
                    [-1, -1]])

# Gradient filter for detecting edges in the opposite direction
mask_t2 = np.array([[1, 1],
                    [1, 1]])

# Laplacian filter for edge enhancement
mask_lap = np.array([[1/12, 1/6, 1/12],
                     [1/6, 0, 1/6],
                     [1/12, 1/6, 1/12]])


**Lucas Kanade**

In [ ]:
def lucas_kanade(fx, fy, ft, frame1):
    # Initialize matrices to store optical flow components (u and v)
    u = np.zeros(frame1.shape)
    v = np.zeros(frame1.shape)

    # Iterate through each pixel in the frame
    for i in range(1, u.shape[0]):
        for j in range(1, u.shape[1]):
            # Extract local gradients around the current pixel
            fxx = fx[i - 1:i + 2, j - 1:j + 2].flatten()
            fyy = fy[i - 1:i + 2, j - 1:j + 2].flatten()
            ftt = ft[i - 1:i + 2, j - 1:j + 2].flatten()

            # Form the A matrix for linear equations Ax = b
            A = np.vstack([fxx, fyy]).T

            # Solve the linear equations using the pseudo-inverse
            res = np.matmul(np.matmul(np.linalg.pinv(np.matmul(A.T, A)), A.T), ftt)

            # Store the estimated optical flow components (u and v)
            u[i, j] = res[0]
            v[i, j] = res[1]

    return u, v
#We have opted for a 3*3 window size search

**Horn Schunck**

In [ ]:
def horn_schunck(fx, fy, ft, frame1):
    # Initialize horizontal and vertical optical flow components
    uh = np.zeros(frame1.shape)
    vh = np.zeros(frame1.shape)

    # Iterative optimization loop (typically 1000 iterations)
    for i in range(1000):
        # Compute the spatial averages of horizontal and vertical flow components using Laplacian filter
        uavg = signal.convolve2d(uh, mask_lap, mode='same')
        vavg = signal.convolve2d(vh, mask_lap, mode='same')

        # Numerator and denominator for updating optical flow components
        num = fx * uavg + fy * vavg + ft
        den = 4 * 15 + fx**2 + fy**2  # The term "4*15" represents the regularization parameter

        # Update optical flow components using the Horn-Schunck equation
        uh = uavg - fx * (num / den)
        vh = vavg - fy * (num / den)

    return uh, vh

**Lucas Kanade Pyramid**

In [ ]:
def lucas_kanade_pyramid(fx, fy, ft, frame1, levels=3):
    u = np.zeros(frame1.shape)
    v = np.zeros(frame1.shape)

    # Iterate over pyramid levels (from coarse to fine)
    for level in range(levels, 0, -1):
        scale = 2 ** (level - 1)

        # Resize the frame to the current level in the pyramid
        scaled_frame1 = cv2.resize(frame1, (0, 0), fx=1 / scale, fy=1 / scale)

        # Iterate through each pixel in the frame
        for i in range(1, u.shape[0]):
            for j in range(1, u.shape[1]):
                # Extract local gradients around the current pixel
                fxx = fx[i - 1:i + 2, j - 1:j + 2].flatten()
                fyy = fy[i - 1:i + 2, j - 1:j + 2].flatten()
                ftt = ft[i - 1:i + 2, j - 1:j + 2].flatten()

                # Form the A matrix for linear equations Ax = b
                A = np.vstack([fxx, fyy]).T

                # Solve the linear equations using the pseudo-inverse
                res = np.matmul(np.matmul(np.linalg.pinv(np.matmul(A.T, A)), A.T), ftt)

                # Update the optical flow components (u and v) at each level
                u[i, j] += res[0]
                v[i, j] += res[1]

    return u, v


# **Defining the EAR and MAR functions**

In [ ]:
def EAR(flow, leftEyeHull, rightEyeHull):
    treye, tleye = [], []

    # Calculate the optical flow vectors for the points in the left eye hull
    for lint in leftEyeHull:
        y, x = lint[0][1], lint[0][0]
        fx, fy = flow[y, x].T
        tleye += [(x + fx, y + fy)]

    # Calculate the optical flow vectors for the points in the right eye hull
    for rint in rightEyeHull:
        y, x = rint[0][1], rint[0][0]
        fx, fy = flow[y, x].T
        treye += [(x + fx, y + fy)]

    # Calculate EAR for the left eye
    earleft = np.linalg.norm([tleye[1][0] - tleye[0][0], tleye[1][1] - tleye[0][1]], ord=1) + \
              np.linalg.norm([tleye[1][0] - tleye[3][0], tleye[1][1] - tleye[3][1]], ord=1)
    earleft /= 2 * np.linalg.norm([tleye[2][0] - tleye[0][0], tleye[2][1] - tleye[0][1]], ord=1)

    # Calculate EAR for the right eye
    earright = np.linalg.norm([treye[1][0] - treye[0][0], treye[1][1] - treye[0][1]], ord=1) + \
               np.linalg.norm([treye[1][0] - treye[3][0], treye[1][1] - treye[3][1]], ord=1)
    earright /= 2 * np.linalg.norm([treye[2][0] - treye[0][0], treye[2][1] - treye[0][1]], ord=1)

    return earleft, earright

In [ ]:
def MAR(flow, mouthHull):
    tm = []

    # Calculate the optical flow vectors for the points in the mouth hull
    for m in mouthHull:
        y, x = m[0][1], m[0][0]
        fx, fy = flow[y, x].T
        tm += [(x + fx, y + fy)]

    # Calculate MAR for the mouth
    mar = np.linalg.norm([tm[1][0] - tm[3][0], tm[1][1] - tm[3][1]])
    mar /= np.linalg.norm([tm[0][0] - tm[2][0], tm[0][1] - tm[2][1]])

    return mar

# **Defining the drowsiness detection function**

In [ ]:
def drowsiness_detection(number, lucas_kanade):
    # Get the list of frames for the given number
    frames = display_files_ordered(folder_path, number)
    # Retrieve facial landmarks for the initial frame
    leftEyeHull, rightEyeHull, mouthHull, jint, c = image_landmarks(number)
    # Read the initial frame
    frame1 = cv2.imread(os.path.join(data_dir, frames[c]), 0)
    frame1 = cv2.resize(frame1, (450, 800))
    state = "Stable"  # Initial state

    # Iterate through subsequent frames
    for frame in frames[c + 1:]:
        # Read the current frame
        frame2 = cv2.imread(os.path.join(data_dir, frame))
        frame2 = cv2.cvtColor(frame2, cv2.COLOR_RGB2GRAY)
        frame2 = cv2.resize(frame2, (450, 800))

        # Apply Gaussian blur to both frames
        frame1 = cv2.GaussianBlur(frame1, (5, 5), 0)
        frame2 = cv2.GaussianBlur(frame2, (5, 5), 0)

        # Calculate spatial and temporal gradients for optical flow
        fx = signal.convolve2d(frame1, mask_x, mode='same') + signal.convolve2d(frame2, mask_x, mode='same')
        fy = signal.convolve2d(frame1, mask_y, mode='same') + signal.convolve2d(frame2, mask_y, mode='same')
        ft = signal.convolve2d(frame1, mask_t1, mode='same') + signal.convolve2d(frame2, mask_t2, mode='same')

        # Calculate optical flow
        u, v = lucas_kanade(fx, fy, ft, frame1)
        flow = np.concatenate((u, v), axis=0).reshape((800, 450, 2))

        # Store previous EAR, MAR values
        pearleft, pearright = earleft, earright
        pmar = mar

        # Calculate current EAR and MAR
        earleft, earright = EAR(flow, leftEyeHull, rightEyeHull, frame1)
        mar = MAR(flow, mouthHull)

        print(earleft, earright, mar)

        # Alert condition based on EAR and MAR thresholds
        if (pearleft < 0.4 and earleft < 0.4) or (pearright < 0.4 and earright < 0.4) or (pmar > 4 and mar > 4):
            state = "Alert"
            return state

    return state


In [ ]:
plt.figure(figsize=(15,15))
plt.imshow(vis)